# Thorough cycle analysis: how place fields form & evolve

Place-field (PF) analysis built on the Gaussian RF fits of the cycles experiment (`results/cycles/gaussian_rf_fits.npz`, produced from `src/cycles/`).

**Part 1 - dead-cell characterization:** a cell is *dead* in a `(room, cycle)` when its `signal_max` (recorded after visiting that room) falls below a threshold.

**Part 2 - room / place-field analysis:** the PF of a cell in a `(room, cycle)` is the *sum of its fitted Gaussians*; we look at how PFs appear and evolve across rooms and cycles.

**Part 3 - extra analyses.**

Every figure is shown inline and saved as a PDF. The number of cycles and rooms is inferred from the results. Computations live in `analysis.dead_cells_analysis` (reusable dead characterization) and `analysis.pf_evolution_analysis`; plotting in `analysis.pf_evolution_plots`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

for _path in (Path.cwd(), *Path.cwd().parents):
    if (_path / "pyproject.toml").is_file():
        for _p in (str(_path), str(_path / "src")):
            if _p not in sys.path:
                sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("Could not find project root (pyproject.toml)")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis.gaussian_fit_cycle_analysis import build_cell_receptive_fields_df
from analysis import dead_cells_analysis as dead
from analysis import pf_evolution_analysis as pfa
from analysis import pf_evolution_plots as pfp
from analysis.pf_evolution_plots import save_figure
from cycles.cycles_paths import CYCLES_PLOTS_BASE, CYCLES_RESULTS_BASE

## Notebook-wide constants

In [ ]:
# --- Input ---
RESULTS_DIR = CYCLES_RESULTS_BASE
GAUSSIANS_RESULTS_PATH = RESULTS_DIR / "gaussian_rf_fits.npz"
assert GAUSSIANS_RESULTS_PATH.is_file(), (
    f"Missing {GAUSSIANS_RESULTS_PATH.resolve()}"
)

# --- Output (figures are saved here as PDF) ---
OUTPUT_DIR = CYCLES_PLOTS_BASE / "thorough_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Dead-cell thresholds (on signal_max) ---
DEAD_THRESHOLD = 0.1
DEAD_THRESHOLDS = (0.1, 0.15, 0.2, 0.25, 0.3, 0.5)

# --- Number of fitted Gaussians, read from the results file ---
with np.load(GAUSSIANS_RESULTS_PATH) as _meta:
    N_GAUSSIANS = int(_meta["n_gaussians"])
print(f"n_gaussians = {N_GAUSSIANS}")


def finalize(fig, name):
    """Show *fig* inline and save it as a PDF under OUTPUT_DIR."""
    out_path = OUTPUT_DIR / f"{name}.pdf"
    save_figure(fig, out_path, close=False)
    plt.show()
    plt.close(fig)
    print(f"saved {out_path}")

In [ ]:
df_rf = build_cell_receptive_fields_df(GAUSSIANS_RESULTS_PATH)

# Cycles and rooms are inferred from the data (granularity-agnostic).
CYCLES, ROOMS = dead.infer_grid_shape(df_rf)
N_CYCLES, N_ROOMS = len(CYCLES), len(ROOMS)
N_CELLS = int(df_rf.index.get_level_values("cell_idx").nunique())
print(
    f"{N_CELLS} cells x {N_CYCLES} cycles x {N_ROOMS} rooms "
    f"(rooms {ROOMS[0]}..{ROOMS[-1]}, cycles {CYCLES[0]}..{CYCLES[-1]})"
)

# Long activity table reused by every dead-threshold computation.
signal_long = dead.build_signal_long_df(df_rf)
signal_long.head()

## 1. Dead-cell analysis

A cell is **dead** in a `(room, cycle)` when its `signal_max` is below the dead threshold (per room, after the cell has visited that room).

### 1a. Binary dead / active table (`cell` x `cycle` x `room`)

In [ ]:
dead_df = dead.build_dead_cells_df(df_rf, dead_threshold=DEAD_THRESHOLD)
print(f"dead fraction @ {DEAD_THRESHOLD}: {dead_df['dead'].mean():.3f}")
dead_df.head()

### 1b. Proportion of dead cells per room, grouped by cycle

x = room, one bar per cycle (colour = cycle), y = proportion of dead cells.

In [ ]:
dead_prop = dead.dead_proportion_by_room_cycle(dead_df)
fig = pfp.plot_dead_proportion_grouped_bars(
    dead_prop,
    title=(
        "Dead-cell proportion per room, grouped by cycle "
        f"(threshold={DEAD_THRESHOLD})"
    ),
)
finalize(fig, "1b_dead_proportion_grouped_bars")

### 1c. Dead proportion across thresholds

One line per `dead_threshold`; each point is the mean over cycles for that room (shaded band = +/-1 std across cycles).

In [ ]:
dead_prop_multi = dead.dead_proportion_multi_threshold(
    signal_long, thresholds=DEAD_THRESHOLDS,
)
fig = pfp.plot_dead_proportion_multithreshold_lines(dead_prop_multi)
finalize(fig, "1c_dead_proportion_multithreshold")

### 1d. Recoveries per cell vs threshold

A *recovery* is a `dead -> active` transition for a `(cell, room)` across consecutive cycles (the cell re-activates for a room after going dead). Counts are summed over rooms, giving one value per cell; one violin per threshold.

In [ ]:
recoveries_multi = dead.recoveries_multi_threshold(
    signal_long, thresholds=DEAD_THRESHOLDS,
)
fig = pfp.plot_recoveries_violin(recoveries_multi)
finalize(fig, "1d_recoveries_violin")

## 2. Room / place-field analysis

The **place field (PF)** of a cell in a `(room, cycle)` is the sum of its fitted Gaussians.

### 2a. Place field across rooms x cycles for 3 active cells

Rows = rooms, columns = cycles, one figure per cell. Dead panels are outlined in red; failed fits are grey.

In [ ]:
N_ACTIVE_CELLS = 3
RENDER_SHAPE = (40, 40)  # PF render resolution for the big grids

selected_cells = pfa.select_active_cells(dead_df, df_rf, n=N_ACTIVE_CELLS)
print(f"selected active cells: {selected_cells}")

cell_pf_grids = {}
for cell_idx in selected_cells:
    grid = pfa.build_cell_pf_grid(
        df_rf,
        cell_idx,
        n_gaussians=N_GAUSSIANS,
        dead_threshold=DEAD_THRESHOLD,
        render_shape=RENDER_SHAPE,
    )
    cell_pf_grids[cell_idx] = grid
    fig = pfp.plot_cell_pf_grid(grid)
    finalize(fig, f"2a_pf_grid_cell{cell_idx}")

### 2b. Number of rooms a cell is active for, per cycle

Within a cycle every room is visited once; we count, per cell, how many rooms it was active for. One violin per cycle (over all cells); the red line is the mean.

In [ ]:
active_rooms = dead.active_rooms_per_cell_cycle(dead_df)
fig = pfp.plot_active_rooms_violin(active_rooms)
finalize(fig, "2b_active_rooms_violin")

### 2c. Weighted place-field descriptor time-series

For each active `(cell, room)` and cycle we collapse the Gaussians into 3 scalars - amplitude, x-centre, y-centre - weighting each Gaussian by its amplitude relative to the others (`w_k = amp_k / sum_j amp_j`, so weights sum to 1).

In [ ]:
weighted_pf = pfa.compute_weighted_pf_df(df_rf, n_gaussians=N_GAUSSIANS)
weighted_pf.head()

#### Time-series for the 3 selected cells (one plot per active cell-room)

Amplitude on the left axis, the two centre coordinates on the right axis. The number of rooms shown per cell is capped for readability (set `MAX_TIMESERIES_ROOMS_PER_CELL = None` to plot all active rooms).

In [ ]:
MAX_TIMESERIES_ROOMS_PER_CELL = 4

active_pairs = dead.active_cell_room_pairs(dead_df, min_active_cycles=2)
for cell_idx in selected_cells:
    rooms_for_cell = active_pairs[active_pairs["cell_idx"] == cell_idx]
    rooms_for_cell = rooms_for_cell.sort_values(
        "n_active_cycles", ascending=False
    )
    if MAX_TIMESERIES_ROOMS_PER_CELL is not None:
        rooms_for_cell = rooms_for_cell.head(MAX_TIMESERIES_ROOMS_PER_CELL)
    for room_id in rooms_for_cell["room_id"]:
        ts = pfa.weighted_pf_timeseries(
            weighted_pf,
            cell_idx,
            int(room_id),
            dead_threshold=DEAD_THRESHOLD,
        )
        fig = pfp.plot_weighted_timeseries(
            ts, cell_idx=cell_idx, room_id=int(room_id)
        )
        finalize(fig, f"2c_timeseries_cell{cell_idx}_room{int(room_id)}")

#### Dispersion of the weighted descriptors (std over cycles)

One std per active `(cell, room)` for each descriptor; the violin shows the distribution across all active cell-rooms. The y-axis is symlog because amplitude and pixel coordinates live on different scales.

In [ ]:
PF_STD_MIN_ACTIVE_CYCLES = 2

pf_std_long, pf_std_wide = pfa.pf_variation_std_table(
    weighted_pf,
    dead_threshold=DEAD_THRESHOLD,
    min_active_cycles=PF_STD_MIN_ACTIVE_CYCLES,
)
print(
    f"{len(pf_std_wide):,} active cell-rooms with "
    f">= {PF_STD_MIN_ACTIVE_CYCLES} active cycles"
)
fig = pfp.plot_pf_std_violin(pf_std_long)
finalize(fig, "2c_pf_std_violin")

## 3. Additional analyses

Further views of how place fields form and evolve over training, with respect to rooms and cycles.

### 3a. Active-cell fraction over room x cycle

Heatmap of `1 - dead_proportion`; reveals which rooms 'fill in' with place cells and how that changes across cycles.

In [ ]:
fig = pfp.plot_active_fraction_heatmap(dead_df)
finalize(fig, "3a_active_fraction_heatmap")

### 3b. Population activity over cycles

Fraction of active `(cell, room)` visits and the number of distinct active cells, per cycle.

In [ ]:
fig = pfp.plot_population_activity_curve(dead_df)
finalize(fig, "3b_population_activity_curve")

### 3c. When place fields first form (onset cycle)

For each `(cell, room)` that ever becomes active, the first cycle it is active in.

In [ ]:
onset = dead.pf_onset_cycle(dead_df)
fig = pfp.plot_onset_cycle_hist(onset)
finalize(fig, "3c_onset_cycle_hist")

### 3d. Place-field centre drift over active cycles

Total path length travelled by the amplitude-weighted PF centre across the cycles a `(cell, room)` is active - how stable vs mobile place fields are.

In [ ]:
center_drift = pfa.weighted_center_drift(
    weighted_pf, dead_threshold=DEAD_THRESHOLD, min_active_cycles=2,
)
fig = pfp.plot_center_drift_hist(center_drift)
finalize(fig, "3d_center_drift_hist")

### 3e. Room generalisation (rooms each active cell fires in)

How many distinct rooms each (ever-active) cell is active in - room-specific vs broadly-tuned cells.

In [ ]:
rooms_active = pfa.rooms_active_per_cell(dead_df)
fig = pfp.plot_rooms_active_hist(rooms_active, n_rooms=N_ROOMS)
finalize(fig, "3e_rooms_active_hist")